In [1]:
import timm

model = timm.create_model(
    'vit_base_patch16_224',
    pretrained=True
)

In [2]:
model

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False

In [3]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

test_dataset = datasets.CIFAR100(
    root="./data",
    train=False,
    transform=transform,
    download=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=4
)

Files already downloaded and verified


/home/qasymjomart/fq-vit/.venv/lib/python3.11/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [4]:
import torch

In [6]:
correct = 0
total = 0

model.cuda()
model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.cuda()
        labels = labels.cuda()

        logits = model(images)
        preds = logits.argmax(1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"Acc = {100 * correct / total:.2f}%")

Acc = 0.40%


In [8]:
correct, total

(40, 10000)

In [7]:
import torch; print(torch.__version__)

2.12.1+cu130


In [10]:
from datasets import load_dataset

ds = load_dataset(
    "ILSVRC/imagenet-1k",
    split="validation",
    streaming=True
)



README.md:   0%|          | 0.00/87.6k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

In [12]:
from timm.data import resolve_model_data_config
from timm.data.transforms_factory import create_transform

# Official preprocessing
data_config = resolve_model_data_config(model)
transform = create_transform(**data_config, is_training=False)

# Dataset
ds = load_dataset(
    "ILSVRC/imagenet-1k",
    split="validation",
    streaming=True,
)

top1_correct = 0
top5_correct = 0
total = 0


Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

In [29]:
from tqdm import tqdm

with torch.no_grad():
    for sample in tqdm(ds):
        image = sample["image"]
        label = sample["label"]
        
        if image.mode != "RGB":
            image = image.convert("RGB")

        x = transform(image).unsqueeze(0).cuda()

        logits = model(x)

        # Top-1
        pred1 = logits.argmax(dim=1).item()
        top1_correct += (pred1 == label)

        # Top-5
        pred5 = logits.topk(5, dim=1).indices.squeeze(0)
        top5_correct += (label in pred5.cpu().tolist())

        total += 1

        if total % 1000 == 0:
            print(
                f"{total}: "
                f"Top1={100*top1_correct/total:.2f}% "
                f"Top5={100*top5_correct/total:.2f}%"
            )

print(f"\nFinal Top-1: {100*top1_correct/total:.2f}%")
print(f"Final Top-5: {100*top5_correct/total:.2f}%")

913it [00:19, 76.33it/s]

1000: Top1=85.50% Top5=96.90%


1914it [00:33, 68.05it/s]

2000: Top1=85.60% Top5=97.05%


2911it [00:47, 68.84it/s]

...

49000: Top1=85.08% Top5=97.53%


49914it [12:50, 75.64it/s]

50000: Top1=85.09% Top5=97.53%


50000it [12:52, 64.76it/s]


Final Top-1: 85.10%
Final Top-5: 97.53%
